# Préparation du TP

### 1. Diagramme pôle-zéro

* **Fréquence d'échantillonnage :** $\nu_e = 20\text{ kHz}$.
* **Zéros :** Positionnés sur le cercle unité pour éliminer les composantes indésirables. Pour garantir des coefficients réels, les zéros complexes sont en paires conjuguées :
    * $z_1, z_2 = e^{\pm j0.2625\pi}$
    * $z_3, z_4 = e^{\pm j0.728\pi}$
    * $z_5 = e^{j\pi} = -1$ (assure le gain nul à 10 kHz ).
* **Pôles :** Le filtre est à réponse impulsionnelle finie (RIF)  et causal. Il possède 5 pôles situés à l'origine du plan complexe ($p_{1..5} = 0$).



In [ ]:
import numpy as np
import scipy.signal as signal
from msicpe.tns import zplane

# Paramètres du signal
nu_e = 20000  # Fréquence d'échantillonnage en Hz
nu_1 = 2625   # Fréquence perturbatrice 1 en Hz
nu_2 = 7280   # Fréquence perturbatrice 2 en Hz
nu_3 = 10000  # Fréquence de coupure en Hz

# Calcul des pulsations numériques normalisées
w1 = 2 * np.pi * nu_1 / nu_e
w2 = 2 * np.pi * nu_2 / nu_e
w3 = 2 * np.pi * nu_3 / nu_e

# Définition des zéros (paires conjuguées + zéro à -1)
zeros = np.array([
    np.exp(1j * w1), np.exp(-1j * w1),
    np.exp(1j * w2), np.exp(-1j * w2),
    np.exp(1j * w3)
])

# Définition des pôles (à l'origine pour un RIF causal)
poles = np.zeros(len(zeros))

# Affichage du diagramme pôle-zéro en utilisant la fonction fournie
zplane(zeros, [])

### 2. Fonction de transfert

La fonction de transfert $H(z)$ ] est le produit des binômes formés par les zéros, ajusté par une constante de gain $K$ :

$$H(z) = K \cdot (1 - z_1z^{-1})(1 - z_2z^{-1})(1 - z_3z^{-1})(1 - z_4z^{-1})(1 - z_5z^{-1})$$

### 3. Obtention de la réponse impulsionnelle

La fonction de transfert d'un filtre RIF s'écrit sous la forme d'un polynôme en puissances négatives de $z$:

$$H(z) = \sum_{k=0}^{N} h[k] z^{-k}$$

Les valeurs de la réponse impulsionnelle $h[k]$ sont les coefficients du polynôme développé de la fonction de transfert.

### 4. Calcul du facteur de gain

Pour obtenir un gain unitaire à la fréquence nulle ($\nu = 0$, soit $z=e^{j0}=1$) :

1. Évaluer la fonction de transfert non normalisée en $z=1$.
2. Appliquer la condition $H(1) = 1$.
3. Le facteur de gain $K$ est l'inverse de la somme des coefficients du filtre non normalisé :

$$K = \frac{1}{\sum_{k=0}^{N} h_{non\_normalise}[k]}$$

# 1 Convolutions linéaire et circulaire

In [3]:
import numpy as np 
import scipy.signal as signal 
import plotly.express as px 
import pandas as pd # pour les dataframes 
from msicpe.tns import zplane

In [10]:
x = np.array([np.abs(k - 5) for k in range(0,11)])
h = np.ones(10)
px.line(x=np.arange(len(x)), y=x, labels={"x":"Indices", "y":"Amplitude"},title='Séquence x[k]').show()
px.line(x=np.arange(len(h)), y=h, labels={"x":"Indices", "y":"Amplitude"},title='Séquence h[k]').show()



In [5]:
convol = np.convolve(x, h, mode='full')
px.line(x=np.arange(len(convol)), y=convol, labels={"x":"Indices", "y":"Amplitude"},title='Produit de convolution de x[k] avec h[k]')

Le produit de convolution obtenu est de longueur 20. On a L = 11 et M = 10 donc L + M - 1 = 20 soit la longueur du produit de convolution.

Pour obtenir cette convolution circulaire il faut faire la TFD de x[k] et de h[k], puis faire leur produit, puis faire la TFD inverse afin d'obtenir cette convolution circulaire.

In [6]:
def convol_circ(x, h, N):
    '''Fonction permettant d'effectuer la convolution circulaire de deux séquences dans le domaine fréquentiel.
    Elle prend en argument deux séquences x[k] et h[k] et également N le nombre de points sur lequel sont calculées les TFD.
    Elle renvoie le résultat de la convolution circulaire noté yc[k]'''
    # Calcul des TFD sur N points (le paramètre n gère le zero-padding)
    X_f = np.fft.fft(x, n=N)
    H_f = np.fft.fft(h, n=N)

    # Produit dans le domaine fréquentiel
    Y_c_f = X_f * H_f
    
    # TFDI pour retour dans le domaine temporel (partie réelle pour éviter les résidus complexes)
    return np.real(np.fft.ifft(Y_c_f))

In [7]:
valeurs_N = [11, 15, 20]

for N in valeurs_N:
    y_c = convol_circ(x, h, N)
    k_yc = np.arange(N)
    # Affichage
    px.line(x=k_yc, y=y_c, labels={'x': 'Indices', 'y': 'Amplitude'}, title=f'Convolution circulaire (N={N})').show()

Lorsque N est inférieur à L + M - 1 il y a un repliement des valeurs excédentaires sur les premiers indices. On retrouve alors une séquence déformée car les valeurs des indices excédentaires s'ajoutent à celles du début.

# 2 Synthèse de filtre par positionnement de pôles et zéros

### 2.1 Synthèse du filtre

In [42]:
# Paramètres du système
nu_e = 20000 
nu_1 = 2625  
nu_2 = 7280  
nu_coupure = 10000

# 1. Calcul des zéros
w1 = 2 * np.pi * nu_1 / nu_e
w2 = 2 * np.pi * nu_2 / nu_e
w_coupure = 2 * np.pi * nu_coupure / nu_e

z1 = np.exp(1j * w1)
z2 = np.exp(1j * w2)
z3 = np.exp(1j * w_coupure) # Correspond à -1

# Regroupement des zéros
zeros = np.array([z1, np.conj(z1), z2, np.conj(z2), z3])

# Filtre RIF causal : 5 pôles à l'origine
poles = np.zeros(len(zeros)) 

# 2. Coefficients de la fonction de transfert
b_non_norm = np.real(np.poly(zeros))
gain_k = np.sum(b_non_norm) 

# Normalisation
b = b_non_norm / gain_k
a = np.array([1.0])

# 3. Calcul du gain complexe
W, h = signal.freqz(b, a, worN=1024, fs=nu_e)

# 4. Tracés
# Diagramme pôle-zéro
zplane(zeros, [])

# Réponse impulsionnelle
px.line(x=np.arange(len(b)), y=b, title='Réponse impulsionnelle', labels={'x': 'Échantillons', 'y': 'Amplitude'}).show()

# Module du gain complexe
gain_db = 20 * np.log10(np.abs(h))
px.line(x=W, y=gain_db, title='Module du gain complexe', labels={'x': 'Fréquence (Hz)', 'y': 'Gain (dB)'}).show()

# Phase du gain complexe
phase = np.angle(h)
px.line(x=W, y=phase, title='Phase du gain complexe', labels={'x': 'Fréquence (Hz)', 'y': 'Phase (rad)'}).show()

### 2.2 Application du filtrage

In [ ]:
# 1. Chargement du signal 4
data = np.load('Signaux/signal4.npz')
s = data['s']

N_points = len(s)
t = np.arange(N_points) / nu_e

# Brouillage additif (amplitude 8)
brouillage = 8 * np.sin(2 * np.pi * nu_1 * t) + 8 * np.sin(2 * np.pi * nu_2 * t)
s_brouille = s + brouillage

# Filtrage du signal
s_filtre = signal.lfilter(b, a, s_brouille)
s_filtre = np.real(s_filtre)

# Tracés temporels
df_temp = pd.DataFrame({
    'Temps (s)': t, 
    'Signal original': s, 
    'Signal brouillé': s_brouille, 
    'Signal filtré': s_filtre
})

px.line(
    df_temp, 
    x='Temps (s)', 
    y=['Signal original', 'Signal brouillé', 'Signal filtré'], 
    title='Signaux temporels'
).show()

# Calcul des spectres d'amplitude via FFT
S_orig = np.abs(np.fft.fft(s))
S_brouille = np.abs(np.fft.fft(s_brouille))
S_filtre_fft = np.abs(np.fft.fft(s_filtre))

# Axe des fréquences et masque (0 à 10 kHz)
freqs_fft = np.fft.fftfreq(N_points, 1/nu_e)
mask_freq = (freqs_fft >= 0) & (freqs_fft <= 10000)

# Tracés fréquentiels
df_freq = pd.DataFrame({
    'Fréquence (Hz)': freqs_fft[mask_freq],
    'Spectre original': S_orig[mask_freq],
    'Spectre brouillé': S_brouille[mask_freq],
    'Spectre filtré': S_filtre_fft[mask_freq]
})

px.line(
    df_freq, 
    x='Fréquence (Hz)', 
    y=['Spectre original', 'Spectre brouillé', 'Spectre filtré'], 
    title='Spectres d\'amplitude (0 à 10 kHz)'
).show()